In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 - Carga y validación de datos
# MAGIC
# MAGIC **Control:** control-2
# MAGIC
# MAGIC Este notebook utiliza directamente la tabla que ya fue cargada manualmente en Databricks:
# MAGIC
# MAGIC `workspace.control2.datos_bizflix_control_2`
# MAGIC
# MAGIC La tabla se valida y se persiste como una tabla Delta de trabajo para el pipeline.

# COMMAND ----------

from pyspark.sql import functions as F

SOURCE_TABLE = "workspace.control2.datos_bizflix_control_2"
RAW_TABLE = "workspace.control2.control2_clientes_raw"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Leer la tabla existente

# COMMAND ----------

df_raw = spark.table(SOURCE_TABLE)

print("Tabla origen:", SOURCE_TABLE)
print("Número de registros:", df_raw.count())
print("Número de columnas:", len(df_raw.columns))

display(df_raw.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Revisar estructura

# COMMAND ----------

df_raw.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Validaciones

# COMMAND ----------

# Nulos
nulls = df_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
])

display(nulls)

# Duplicados por cliente
if "cliente_id" in df_raw.columns:
    duplicados = (
        df_raw.groupBy("cliente_id")
        .count()
        .filter(F.col("count") > 1)
    )
    print("Clientes duplicados:", duplicados.count())

# Distribución de la variable objetivo
if "supera_50k" in df_raw.columns:
    display(
        df_raw.groupBy("supera_50k")
        .count()
        .withColumn(
            "porcentaje",
            F.round(F.col("count") / df_raw.count() * 100, 2)
        )
        .orderBy("supera_50k")
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Limpieza y tipado

# COMMAND ----------

df_clean = df_raw.dropDuplicates(["cliente_id"])

# Conversión de tipos solo si las columnas existen
casts = {
    "edad": "int",
    "antiguedad_laboral_anios": "double",
    "experiencia_anios": "double",
    "ingreso_anual_usd": "double",
    "deuda_mensual_usd": "double",
    "score_crediticio": "int",
    "productos_activos": "int",
    "antiguedad_cliente_meses": "int",
    "transacciones_mensuales": "int",
    "saldo_promedio_usd": "double",
    "mora_12m": "int",
    "supera_50k": "int",
    "cliente_premium": "int"
}

for col_name, data_type in casts.items():
    if col_name in df_clean.columns:
        df_clean = df_clean.withColumn(
            col_name,
            F.col(col_name).cast(data_type)
        )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Persistir como Delta

# COMMAND ----------

(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_TABLE)
)

print("Tabla Delta creada/actualizada:", RAW_TABLE)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Verificación final

# COMMAND ----------

df_delta = spark.table(RAW_TABLE)

print("Registros en tabla Delta:", df_delta.count())
display(df_delta.limit(10))
